# Deliverable A Underwriting Policy

This notebook turns the hackathon brief and the local markdown guidance into a runnable Deliverable A workflow:

1. audit the statistical assumptions and data gaps,
2. train a calibrated probability-of-default model,
3. construct 90% PD intervals,
4. convert PD into approve/reject decisions using expected NPV,
5. write `outputs/submission/submission_A_decisions.csv`.

The reusable implementation lives in `src/deliverable_a_pipeline.py`; this notebook keeps the analysis and execution path visible cell by cell.

## Statistical Audit Summary

The updated project files consistently recommend this framing:

- Deliverable A is not plain classification; it is a calibrated PD estimate plus a lending policy.
- The equation in the brief implies `decision = 1[E[NPV | approve] > 0]`, not `PD < fixed threshold`.
- Missing `default_flag` values are unknown outcomes, not non-default labels.
- Prior-underwriter fields can be predictive but should not be treated as causal drivers.
- Intervals should reflect model dispersion, calibration uncertainty, and support/selection risk.

Important gap found against the CSVs: the markdown says validation is fully labeled, but the data shows validation labels are also only present for prior-approved rows. This makes reject inference the main unresolved statistical risk.

## Causal Trap And Middle-Trick Fixes

The added `markdown_files/causal_trap_and_middle_trick_analysis.md` sharpens the core statistical risk: the model trained on observed outcomes estimates `P(default | prior lender approved)`, while Deliverable A asks us to decide over validation/test applicants that include prior declines.

Fixes incorporated here:

- keep a prior-approval propensity model for support diagnostics and interval widening,
- avoid using `prior_decision` and `prior_approved_amount` as PD features,
- keep `prior_underwriter_score` only as a predictive benchmark/proxy and run an ablation without it,
- add structural missingness indicators for bank-feed fields,
- use segment-aware calibration features from prior score, bank-feed presence, prior-approval propensity, and application time,
- keep the engineered confounder features experimental unless they beat the baseline feature set.

The segment-aware calibration can improve ranking/AP in some ablations, but it is monitored against log loss, Brier, and ECE because calibration matters for NPV decisions.

In [ ]:
from pathlib import Path
import os
import sys
import json

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

os.environ.setdefault("DELIVERABLE_A_FEATURE_SET", "baseline")
os.environ.setdefault("DELIVERABLE_A_SEGMENT_META", "1")

from src.deliverable_a_pipeline import (
    CSV_DIR,
    DATA_DIR,
    GROSS_MARGIN_RATE,
    REPORT_DIR,
    SUBMISSION_DIR,
    add_application_features,
    build_intervals,
    calibration_table,
    ensemble_predictions,
    expected_profit_frame,
    feature_columns,
    fit_calibrated_models,
    fit_segment_meta_calibrator,
    apply_segment_meta_calibrator,
    metric_summary,
    realized_profit_proxy,
    train_approval_propensity,
    tune_buffer,
    write_audit_report,
)

REPORT_DIR.mkdir(parents=True, exist_ok=True)
SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)
print(PROJECT_ROOT)

## Load Data

Outcomes are observed only when `default_flag` is not null. We keep that distinction explicit throughout the notebook.

In [ ]:
train = pd.read_csv(CSV_DIR / "train.csv")
validation = pd.read_csv(CSV_DIR / "validation.csv")
test = pd.read_csv(CSV_DIR / "test.csv")

def split_summary_row(name, df):
    observed = df["default_flag"].notna()
    return {
        "split": name,
        "rows": len(df),
        "prior_approval_rate": float((df["prior_decision"] == 1).mean()),
        "observed_outcome_rate": float(observed.mean()),
        "default_rate_when_observed": float(df.loc[observed, "default_flag"].mean()) if observed.any() else np.nan,
        "bank_feed_link_rate": float(df["has_linked_bank_feed"].mean()),
    }

split_summary = pd.DataFrame([
    split_summary_row("train", train),
    split_summary_row("validation", validation),
    split_summary_row("test", test),
])
split_summary

## Feature Policy

We use application-time variables, engineered ratios, time features, and missingness indicators. We drop outcome/post-outcome columns. For the PD model, we also exclude `prior_decision` and `prior_approved_amount` because labeled rows are all prior-approved and those fields would encode selection rather than default risk. We keep `prior_underwriter_score` because it is available for all rows and is a useful predictive risk score.

## Confounder-Derived Feature Engineering

The synthetic data has strongly correlated latent clusters, so the feature builder now creates concept-level features rather than relying only on raw columns:

- `repayment_burden_index`: requested amount relative to stated/observed revenue and debt load.
- `credit_stress_index`: utilization, inquiries, multi-lender behavior, debt burden, and prior default rate.
- `cash_stress_index`: delinquency, utilization, volatility, overdrafts, negative cash, and payroll regularity.
- `maturity_index`: vintage, stated time in business, and account age.
- `revenue_scale_index`: stated and observed revenue scale.
- `platform_engagement_index`: platform activity, prior loan history, and bookkeeping recency.
- `bank_feed_support_index`: bank-feed availability and missingness support.
- `selection_support_index`: prior-policy support proxy, used predictively but interpreted carefully because it reflects selection.

The model also includes interactions identified by the confounder/correlation audit, such as utilization x delinquency and utilization x repayment burden.

In [ ]:
train_fe = add_application_features(train)
validation_fe = add_application_features(validation)
test_fe = add_application_features(test)

_, numeric, categorical = feature_columns(train_fe)
train_x = train_fe[numeric + categorical]
validation_x = validation_fe[numeric + categorical]
test_x = test_fe[numeric + categorical]

print(f"numeric features: {len(numeric)}")
print(f"categorical features: {len(categorical)}")
print(categorical)

## Prior-Approval Propensity

This is not a complete fix for reject inference, but it tells us where the PD model is extrapolating away from the historical approved population. We use it for inverse-propensity weighting during training and for widening intervals when support is weak.

In [ ]:
propensity_model = train_approval_propensity(
    train_x,
    (train["prior_decision"] == 1).astype(int),
    numeric,
    categorical,
)
prop_train = np.clip(propensity_model.predict_proba(train_x)[:, 1], 0.001, 0.999)
prop_validation = np.clip(propensity_model.predict_proba(validation_x)[:, 1], 0.001, 0.999)
prop_test = np.clip(propensity_model.predict_proba(test_x)[:, 1], 0.001, 0.999)

pd.DataFrame({
    "split": ["train", "validation", "test"],
    "mean_propensity": [prop_train.mean(), prop_validation.mean(), prop_test.mean()],
    "p05": [np.quantile(prop_train, 0.05), np.quantile(prop_validation, 0.05), np.quantile(prop_test, 0.05)],
    "p95": [np.quantile(prop_train, 0.95), np.quantile(prop_validation, 0.95), np.quantile(prop_test, 0.95)],
})

## Train Calibrated PD Ensemble

The environment does not currently have CatBoost, LightGBM, or XGBoost installed, so this notebook uses sklearn histogram gradient boosting variants plus a logistic baseline. Each model is calibrated with isotonic regression on a time-based holdout from labeled training rows.

In [ ]:
labeled_train = train[train["default_flag"].notna()].copy()
models, model_idx, cal_idx = fit_calibrated_models(
    train_x,
    labeled_train,
    numeric,
    categorical,
    prop_train,
)

[m.name for m in models], len(model_idx), len(cal_idx)

## Calibration and Validation Metrics

Metrics are computed only on rows with known `default_flag`. Validation is therefore `validation_labeled_only`, not the full validation file.

In [ ]:
cal_x = train_x.loc[cal_idx]
cal_y = train.loc[cal_idx, "default_flag"].astype(int).to_numpy()
cal_point, cal_matrix = ensemble_predictions(models, cal_x)
cal_table = calibration_table(cal_y, cal_point, n_bins=10)
cal_table.to_csv(REPORT_DIR / "deliverable_a_calibration_bins_train_calibration.csv", index=False)

validation_point, validation_matrix = ensemble_predictions(models, validation_x)
test_point, test_matrix = ensemble_predictions(models, test_x)

segment_meta_enabled = os.getenv("DELIVERABLE_A_SEGMENT_META", "0") == "1"
if segment_meta_enabled:
    segment_calibrator, day_center, day_scale = fit_segment_meta_calibrator(
        cal_point,
        train_fe.loc[cal_idx],
        prop_train[cal_idx],
        cal_y,
    )
    cal_point, cal_matrix = apply_segment_meta_calibrator(
        segment_calibrator, cal_point, cal_matrix, train_fe.loc[cal_idx], prop_train[cal_idx], day_center, day_scale
    )
    validation_point, validation_matrix = apply_segment_meta_calibrator(
        segment_calibrator, validation_point, validation_matrix, validation_fe, prop_validation, day_center, day_scale
    )
    test_point, test_matrix = apply_segment_meta_calibrator(
        segment_calibrator, test_point, test_matrix, test_fe, prop_test, day_center, day_scale
    )
    cal_table = calibration_table(cal_y, cal_point, n_bins=10)
    cal_table.to_csv(REPORT_DIR / "deliverable_a_calibration_bins_train_calibration.csv", index=False)

val_labeled_mask = validation["default_flag"].notna()
metrics = {
    "train_calibration_holdout": metric_summary(cal_y, cal_point),
    "validation_labeled_only": metric_summary(
        validation.loc[val_labeled_mask, "default_flag"].astype(int).to_numpy(),
        validation_point[val_labeled_mask.to_numpy()],
    ),
}
metrics["settings"] = {
    "feature_set": os.getenv("DELIVERABLE_A_FEATURE_SET", "baseline"),
    "segment_meta_calibration": segment_meta_enabled,
}
print(json.dumps(metrics, indent=2))
cal_table

## Estimate Loan Economics

The brief gives fixed product terms: 3% origination fee, 35% APR, 60-day term. Recovery is estimated from known train defaults.

The local NPV gap is that the exact official realized-profit formula is not available. This notebook uses a transparent proxy:

- paid loan profit = upfront fee + 60-day simple interest,
- default loss = principal - recovered amount - upfront fee.

In [ ]:
defaulted_train = train[train["default_flag"] == 1]
recovery_rate = float(
    (defaulted_train["final_recovered_amount"] / defaulted_train["requested_amount"])
    .clip(lower=0, upper=1)
    .mean()
)
loss_rate_after_fee = max(0.0, 1.0 - recovery_rate - 0.03)

pd.DataFrame([{
    "gross_margin_rate_if_paid": GROSS_MARGIN_RATE,
    "mean_default_recovery_rate": recovery_rate,
    "loss_rate_after_fee": loss_rate_after_fee,
}])

## Tune Expected-NPV Buffer

The decision rule is `approve if expected_profit > buffer_rate * requested_amount`. The buffer is tuned on labeled validation rows using a realized-profit proxy. This is not perfect because declined validation rows are unlabeled, but it is better aligned with the scoring objective than F1 or accuracy.

In [ ]:
buffer_table = tune_buffer(
    validation.loc[val_labeled_mask].copy(),
    validation_point[val_labeled_mask.to_numpy()],
    loss_rate_after_fee,
)
buffer_table.to_csv(REPORT_DIR / "deliverable_a_buffer_tuning.csv", index=False)
selected_buffer_rate = float(buffer_table.iloc[0]["buffer_rate"])
print("selected_buffer_rate", selected_buffer_rate)
buffer_table

## Build 90% PD Intervals

Intervals combine:

- ensemble 5th/95th percentile spread,
- calibration-bin uncertainty from the train calibration holdout,
- support widening from prior-approval propensity.

In [ ]:
val_lower, val_upper = build_intervals(
    validation_point,
    validation_matrix,
    cal_table,
    prop_validation,
)
test_lower, test_upper = build_intervals(
    test_point,
    test_matrix,
    cal_table,
    prop_test,
)

pd.DataFrame({
    "split": ["validation", "test"],
    "mean_pd": [validation_point.mean(), test_point.mean()],
    "mean_interval_width": [np.mean(val_upper - val_lower), np.mean(test_upper - test_lower)],
    "min_lower": [val_lower.min(), test_lower.min()],
    "max_upper": [val_upper.max(), test_upper.max()],
})

## Convert PD to Approve/Reject by Expected NPV

This implements the slide equation: the final decision uses the expected NPV sign, adjusted by the selected safety buffer, not a flat PD threshold.

In [ ]:
validation_profit = expected_profit_frame(validation, validation_point, loss_rate_after_fee)
test_profit = expected_profit_frame(test, test_point, loss_rate_after_fee)

val_decision = (
    validation_profit["expected_profit"].to_numpy()
    > selected_buffer_rate * validation["requested_amount"].to_numpy(float)
).astype(int)
test_decision = (
    test_profit["expected_profit"].to_numpy()
    > selected_buffer_rate * test["requested_amount"].to_numpy(float)
).astype(int)

diagnostics = pd.DataFrame({
    "split": ["validation", "test"],
    "rows": [len(validation), len(test)],
    "approval_rate": [float(val_decision.mean()), float(test_decision.mean())],
    "mean_predicted_pd": [float(validation_point.mean()), float(test_point.mean())],
    "mean_predicted_pd_approved": [
        float(validation_point[val_decision == 1].mean()) if val_decision.sum() else np.nan,
        float(test_point[test_decision == 1].mean()) if test_decision.sum() else np.nan,
    ],
    "mean_interval_width": [float(np.mean(val_upper - val_lower)), float(np.mean(test_upper - test_lower))],
    "mean_prior_approval_propensity": [float(prop_validation.mean()), float(prop_test.mean())],
    "expected_profit_total": [
        float(validation_profit.loc[val_decision == 1, "expected_profit"].sum()),
        float(test_profit.loc[test_decision == 1, "expected_profit"].sum()),
    ],
})
diagnostics.to_csv(REPORT_DIR / "deliverable_a_submission_diagnostics.csv", index=False)
diagnostics

## Write Submission A

The required row set is validation plus test applicants. The notebook enforces interval ordering and probability bounds before writing.

In [ ]:
submission_a = pd.concat(
    [
        pd.DataFrame({
            "applicant_id": validation["applicant_id"],
            "decision": val_decision,
            "predicted_pd": validation_point,
            "pd_lower_90": val_lower,
            "pd_upper_90": val_upper,
        }),
        pd.DataFrame({
            "applicant_id": test["applicant_id"],
            "decision": test_decision,
            "predicted_pd": test_point,
            "pd_lower_90": test_lower,
            "pd_upper_90": test_upper,
        }),
    ],
    ignore_index=True,
)

submission_a["predicted_pd"] = submission_a["predicted_pd"].clip(0, 1)
submission_a["pd_lower_90"] = np.minimum(submission_a["pd_lower_90"].clip(0, 1), submission_a["predicted_pd"])
submission_a["pd_upper_90"] = np.maximum(submission_a["pd_upper_90"].clip(0, 1), submission_a["predicted_pd"])
submission_a.to_csv(SUBMISSION_DIR / "submission_A_decisions.csv", index=False)

assert len(submission_a) == len(validation) + len(test)
assert submission_a["applicant_id"].is_unique
assert submission_a["decision"].isin([0, 1]).all()
assert (submission_a["pd_lower_90"] <= submission_a["predicted_pd"]).all()
assert (submission_a["predicted_pd"] <= submission_a["pd_upper_90"]).all()
assert submission_a[["predicted_pd", "pd_lower_90", "pd_upper_90"]].between(0, 1).all().all()

submission_a.head()

## Write Statistical Audit Report

This report captures the concepts used, the data-guidance mismatch, and the remaining statistical risks.

In [ ]:
split_summary.to_csv(REPORT_DIR / "deliverable_a_split_summary.csv", index=False)
(REPORT_DIR / "deliverable_a_metrics.json").write_text(json.dumps(metrics, indent=2))
write_audit_report(
    REPORT_DIR / "deliverable_a_statistical_audit.md",
    split_summary,
    metrics,
    selected_buffer_rate,
    recovery_rate,
    loss_rate_after_fee,
)

print("submission:", SUBMISSION_DIR / "submission_A_decisions.csv")
print("audit report:", REPORT_DIR / "deliverable_a_statistical_audit.md")
print("diagnostics:", REPORT_DIR / "deliverable_a_submission_diagnostics.csv")

## Remaining Work

This notebook implements Deliverable A only. The next pieces should use the same calibrated PD outputs:

- Deliverable B: aggregate default timing over the approved applicants by cohort and loan age.
- Deliverable C: build a causal-safe model without prior-underwriter proxies and apply the intervention queries.
- Deliverable D: use the audit report as the technical backbone for the writeup.

## Causal Trap Ablation Result

The causal-trap ablation report is written to `outputs/reports/deliverable_a_causal_trap_ablation.csv`. In the latest script run, segment-aware calibration was enabled for training, but the validation AUROC remained effectively flat versus the original baseline. This is useful evidence: the main causal trap is selection/support risk, not a simple feature-engineering unlock.

## Raw-Score AUROC Experiments

To improve ranking quality, we installed CatBoost and LightGBM, then trained a raw-score experiment outside the calibrated submission pipeline. The important distinction is:

- AUROC should be evaluated on raw ranking scores.
- PD/NPV still needs calibrated probabilities afterward.

The experiment results are saved in:

- `outputs/reports/raw_score_ensemble_experiment.csv`
- `outputs/reports/logistic_tuning_experiment.csv`

The strongest result so far is a regularized logistic model that excludes `prior_underwriter_score`. That is statistically interesting: the prior score is useful for support/selection diagnostics, but removing it improved validation ranking slightly, likely because it reduced imitation of the prior underwriter selection gate.

In [ ]:
raw_score_results = pd.read_csv(REPORT_DIR / "raw_score_ensemble_experiment.csv")
logistic_results = pd.read_csv(REPORT_DIR / "logistic_tuning_experiment.csv")

print("Raw-score ensemble top rows")
display(raw_score_results.sort_values(["auroc", "average_precision"], ascending=False).head(10))

print("Logistic tuning top rows")
display(logistic_results.sort_values(["auroc", "average_precision"], ascending=False).head(10))

## Current Best AUROC

The best validation AUROC from the latest sweep is:

```text
0.757504
```

Configuration:

```text
model: LogisticRegression
feature_set: baseline_no_prior_score
C: 0.06
class_weight: None
score_type: raw probability score
```

This improves over the earlier calibrated ensemble baseline of about `0.754979`, but the improvement is modest. The next ranking experiments should focus on stacked logistic + CatBoost/LightGBM residuals, not more broad hand-built confounder indices.